# Step 2 — Predict Overrun Minutes

Goal: predict `overrun_min` (how late a maintenance block will run) from the features built in Step 1.

**Input required (same folder):** `07_model_training_table.csv`

**Split strategy:** train on 2023–2024, test on 2025. This is a *time-based* split, not random — it mimics how the model will actually be used: predicting overruns for blocks that haven't happened yet, using only what was known in the past.


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib

df = pd.read_csv("07_model_training_table.csv", parse_dates=["date"])
print(df.shape)
df.head()


(15000, 21)


,block_id,section_id,date,day_of_week,month,is_weekend,maintenance_type,required_asset_type,priority,weather,...,trains_count,traffic_level,assets_of_type_in_section,asset_type_availability_pct,had_availability_record,section_historical_overrun_rate,planned_duration_min,overrun_min,overrun_flag,actual_duration_min
0,B03841,BAM-PSA,2023-01-01,Sunday,1,1,Ballast Renewal,Ballast Regulator,Low,Light Rain,...,2.0,LOW,0.0,0.0,1,0.557272,221.8,0.2,0,222.0
1,B12744,BE-SPN,2023-01-01,Sunday,1,1,Track Stabilization,Dynamic Track Stabilizer,Low,Clear,...,6.0,MEDIUM,0.0,1.0,1,0.555171,184.3,24.7,1,209.0
2,B08132,BMG-DTK,2023-01-01,Sunday,1,1,Ballast Renewal,Ballast Regulator,Medium,Heavy Rain,...,0.0,LOW,0.0,0.0,1,0.553304,210.2,16.7,1,226.8
3,B08111,G-NGP,2023-01-01,Sunday,1,1,Signal Maintenance,Tower Wagon,Low,Heavy Rain,...,6.0,MEDIUM,0.0,1.0,1,0.554566,116.9,44.7,1,161.6
4,B09264,KOG-RIS,2023-01-01,Sunday,1,1,Signal Maintenance,Tower Wagon,Medium,Extreme Heat,...,1.0,LOW,0.0,1.0,1,0.555311,99.4,16.5,1,116.0


## Drop leakage columns

- `actual_duration_min` = `planned_duration_min + overrun_min` — directly encodes the answer, must go.
- `overrun_flag` is also derived straight from `overrun_min` — also excluded from features (it's a separate target for classification, not an input).


In [2]:
LEAKAGE_COLS = ["actual_duration_min", "overrun_flag"]
TARGET = "overrun_min"
feature_df = df.drop(columns=LEAKAGE_COLS)

CATEGORICAL = ["day_of_week", "maintenance_type", "required_asset_type", "priority", "weather", "traffic_level"]
NUMERIC = ["month", "is_weekend", "start_hour", "trains_count", "assets_of_type_in_section",
           "asset_type_availability_pct", "had_availability_record",
           "section_historical_overrun_rate", "planned_duration_min"]


## Time-based train/test split, and one-hot encode categoricals


In [3]:
train_mask = df["date"] < "2025-01-01"
test_mask = ~train_mask
print(f"Train: {train_mask.sum()} rows (2023-2024) | Test: {test_mask.sum()} rows (2025)")

X = pd.get_dummies(feature_df[CATEGORICAL + NUMERIC], columns=CATEGORICAL)
y = feature_df[TARGET]

X_train, X_test = X[train_mask].reset_index(drop=True), X[test_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[test_mask].reset_index(drop=True)


Train: 9990 rows (2023-2024) | Test: 5010 rows (2025)


## Baselines: naive mean, then Linear Regression

Always compare against a "dumb" baseline first — if your model can't beat "just predict the average overrun for every block," it isn't learning anything.


In [4]:
naive_pred = np.full_like(y_test, y_train.mean(), dtype=float)
print("Naive baseline MAE:", round(mean_absolute_error(y_test, naive_pred), 2), "min")

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print("\nLinear Regression:")
print(" MAE:", round(mean_absolute_error(y_test, lr_pred), 2), "min")
print(" RMSE:", round(mean_squared_error(y_test, lr_pred)**0.5, 2), "min")
print(" R2:", round(r2_score(y_test, lr_pred), 3))


Naive baseline MAE: 16.5 min

Linear Regression:
 MAE: 15.78 min
 RMSE: 19.48 min
 R2: 0.082


## XGBoost


In [5]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print("XGBoost:")
print(" MAE:", round(mean_absolute_error(y_test, xgb_pred), 2), "min")
print(" RMSE:", round(mean_squared_error(y_test, xgb_pred)**0.5, 2), "min")
print(" R2:", round(r2_score(y_test, xgb_pred), 3))


XGBoost:
 MAE: 15.96 min
 RMSE: 19.69 min
 R2: 0.062


## Read the results honestly

If R² is low (well under 0.3) and XGBoost barely beats the naive baseline, **that's an expected and honest result here** — not a bug in your pipeline. The synthetic data generator deliberately included a large random noise component in `overrun_min` (real-world overruns genuinely do have irreducible randomness: an unexpected obstruction, a worker calling in sick, etc.), so there's a real ceiling on how predictable it can be.

**What this means for your report:** state the baseline vs. model comparison plainly, and note that a real deployment would need real historical block data to know the *true* predictability ceiling — synthetic noise here is a stand-in, not a promise about how predictable real overruns are.

**What still makes this a legitimate ML component:** the model isn't required to be highly accurate to be useful in the optimizer (Step 3) — even a modest, better-than-naive overrun estimate is enough to prefer safer time windows over riskier ones.


In [6]:
importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 features:")
importances.head(10)


Top 10 features:


weather_Heavy Rain                              0.201186
weather_Fog                                     0.133130
weather_Extreme Heat                            0.036935
weather_Clear                                   0.034652
weather_Light Rain                              0.030427
required_asset_type_Dynamic Track Stabilizer    0.019634
maintenance_type_OHE Maintenance                0.018947
maintenance_type_Track Repair                   0.017754
priority_Medium                                 0.017337
planned_duration_min                            0.017144
dtype: float32

## Save the model

We save both the trained model and the exact column order used — Step 3 (the optimizer) needs to build inputs in this exact same shape to get predictions.


In [7]:
joblib.dump(xgb_model, "overrun_model.joblib")
joblib.dump(list(X.columns), "model_feature_columns.joblib")
print("Saved: overrun_model.joblib, model_feature_columns.joblib")


Saved: overrun_model.joblib, model_feature_columns.joblib
